<a href="https://colab.research.google.com/github/fmaignacio/observatorio-tere/blob/main/organiza_transcricoes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import re
import pandas as pd
from google.colab import drive
from datetime import datetime

# 1. Monte seu Google Drive
drive.mount('/content/drive')

# --- CONFIGURAÇÃO ---
pasta_transcricoes = '/content/drive/MyDrive/observatorio_teresopolis/TranscriçõesSessoesTeresopolis'

LISTA_MESTRA_VEREADORES = [
    "Amanda", "André do Gás", "Bruninho Almeida", "Cacau Repórter", "Caio Perfister",
    "Calé", "Dudu do Resgate", "Diego Barbosa", "Fabinho Filé", "Fidel Faria",
    "Igor Faraco", "João Miguel", "Luciano Santos", "Márcia Valentim", "Marcos Rangel",
    "Maurício Lopes", "Paulinho Nogueira", "Amurim", "Sandrinho", "Totó", "Vitinho Nogueira",
    "Érica Marra", "Totó Online"
]

# Normalização de nomes de vereadores
NORMALIZACAO_VEREADORES = {
    'Vitim Nogueira': 'Vitinho Nogueira',
    'Vitin Nogueira': 'Vitinho Nogueira',
    'Vitinho': 'Vitinho Nogueira',
    'Dudo Resgate': 'Dudu do Resgate',
    'Dudu do': 'Dudu do Resgate',
    'Caio Perfester': 'Caio Perfister',
    'Caio Perfist': 'Caio Perfister',
    'Caio Perfiste': 'Caio Perfister',
    'Caio Perfisti': 'Caio Perfister',
    'Caip Fister': 'Caio Perfister',
    'Ico Faraco': 'Igor Faraco',
    'Cacau': 'Cacau Repórter',
    'Fidel': 'Fidel Faria',
    'Luciano': 'Luciano Santos',
    'João': 'João Miguel',
    'Rangel': 'Marcos Rangel',
    'Maurício': 'Maurício Lopes',
    'Totó Online': 'Totó',
    'Total online': 'Totó',
    'professora Amanda': 'Amanda',
    'professor Amanda': 'Amanda',
}

dados_extraidos = []
print(f"🔍 Iniciando análise COMPLETA com extração MELHORADA de EMENTAS e LINKS\n")

# --- FUNÇÕES AUXILIARES ---

def extrair_url_youtube(texto):
    """Extrai a URL do YouTube do arquivo de transcrição"""
    match = re.search(r'https://www\.youtube\.com/watch\?v=([a-zA-Z0-9_-]{11})', texto)
    if match:
        return match.group(0)
    return None

def extrair_ementa_melhorada(texto_completo, pl_numero, posicao_mencao):
    """
    Extração MELHORADA de ementa - versão mais robusta
    Usa padrões mais greedy e múltiplas estratégias
    """
    # Pega contexto MAIOR ao redor da menção do PL
    inicio = max(0, posicao_mencao - 200)
    fim = min(len(texto_completo), posicao_mencao + 2000)
    contexto = texto_completo[inicio:fim]
    
    # Normaliza espaços e quebras de linha no contexto
    contexto_limpo = re.sub(r'\s+', ' ', contexto)
    
    # ESTRATÉGIA 1: Padrões mais GREEDY para capturar mais texto
    padroes_greedy = [
        # "dispõe sobre" até "e dá outras providências" ou até um ponto final seguido de maiúscula
        r'[Dd]ispõe\s+sobre\s+(.+?)(?:\s+e\s+d[aáà]\s+outras\s+provid[êe]ncias|\.(?:\s+[A-Z]|\s*$))',
        
        # "institui" até fim similar
        r'[Ii]nstitui\s+(.+?)(?:\s+e\s+d[aáà]\s+outras\s+provid[êe]ncias|\.(?:\s+[A-Z]|\s*$))',
        
        # "autoriza" até fim similar  
        r'[Aa]utoriza\s+(.+?)(?:\s+e\s+d[aáà]\s+outras\s+provid[êe]ncias|\.(?:\s+[A-Z]|\s*$))',
        
        # "cria" até fim similar
        r'[Cc]ria\s+(.+?)(?:\s+e\s+d[aáà]\s+outras\s+provid[êe]ncias|\.(?:\s+[A-Z]|\s*$))',
        
        # "altera" até fim similar
        r'[Aa]ltera\s+(.+?)(?:\s+e\s+d[aáà]\s+outras\s+provid[êe]ncias|\.(?:\s+[A-Z]|\s*$))',
        
        # "revoga" até fim similar
        r'[Rr]evoga\s+(.+?)(?:\s+e\s+d[aáà]\s+outras\s+provid[êe]ncias|\.(?:\s+[A-Z]|\s*$))',
        
        # "inclui" até fim similar
        r'[Ii]nclui\s+(.+?)(?:\s+e\s+d[aáà]\s+outras\s+provid[êe]ncias|\.(?:\s+[A-Z]|\s*$))',
        
        # "denomina" padrão específico
        r'[Dd]enomina(?:ção)?\s+(.+?)(?:\s+e\s+d[aáà]\s+outras\s+provid[êe]ncias|\.(?:\s+[A-Z]|\s*$))',
    ]
    
    # ESTRATÉGIA 2: Captura após "de autoria do vereador X" seguido de descrição
    padrao_autoria = r'de\s+autoria\s+d[oa]\s+vereador[a]?\s+[\w\s]+[.,]\s*(.+?)(?:\s+e\s+d[aáà]\s+outras\s+provid[êe]ncias|\.(?:\s+[A-Z]|\s*$))'
    
    # Tenta padrões greedy primeiro
    for padrao in padroes_greedy:
        match = re.search(padrao, contexto_limpo, re.IGNORECASE | re.DOTALL)
        if match:
            ementa_bruta = match.group(0)  # Pega todo o match incluindo "dispõe sobre"
            
            # Limpa a ementa
            ementa = ementa_bruta.strip()
            ementa = re.sub(r'\s+', ' ', ementa)
            
            # Remove "e dá outras providências" do final se presente
            ementa = re.sub(r'\s+e\s+d[aáà]\s+outras\s+provid[êe]ncias\.?$', '', ementa, flags=re.IGNORECASE)
            
            # Limita tamanho mas com limite maior
            if len(ementa) > 500:
                ementa = ementa[:500] + "..."
            
            # Valida que a ementa tem conteúdo significativo (mais de 20 caracteres)
            if len(ementa) > 20:
                return ementa[0].upper() + ementa[1:]
    
    # Tenta padrão de autoria
    match = re.search(padrao_autoria, contexto_limpo, re.IGNORECASE | re.DOTALL)
    if match:
        ementa = match.group(1).strip()
        ementa = re.sub(r'\s+', ' ', ementa)
        if len(ementa) > 500:
            ementa = ementa[:500] + "..."
        if len(ementa) > 20:
            return ementa[0].upper() + ementa[1:]
    
    # ESTRATÉGIA 3: Fallback - captura texto simples após palavras-chave
    padroes_simples = [
        r'[Dd]ispõe\s+sobre\s+(\S+(?:\s+\S+){0,30})',  # Captura até 30 palavras
        r'[Ii]nstitui\s+(\S+(?:\s+\S+){0,30})',
        r'[Aa]utoriza\s+(\S+(?:\s+\S+){0,30})',
        r'[Cc]ria\s+(\S+(?:\s+\S+){0,30})',
    ]
    
    for padrao in padroes_simples:
        match = re.search(padrao, contexto_limpo, re.IGNORECASE)
        if match:
            ementa = match.group(0).strip()
            ementa = re.sub(r'\s+', ' ', ementa)
            if len(ementa) > 500:
                ementa = ementa[:500] + "..."
            if len(ementa) > 15:
                return ementa[0].upper() + ementa[1:]
    
    return "Ementa não identificada"

def normalizar_autor(autor):
    """Normaliza o nome do autor"""
    if not autor or autor == 'nan':
        return 'Autor não identificado'
    
    # Remove texto extra
    for padrao in ['altera a lei', 'institui o programa', 'dispõe sobre']:
        if padrao in autor.lower():
            autor = autor.split(padrao)[0].strip()
    
    # Aplica normalização
    if autor in NORMALIZACAO_VEREADORES:
        return NORMALIZACAO_VEREADORES[autor]
    
    for variante, correto in NORMALIZACAO_VEREADORES.items():
        if variante.lower() in autor.lower():
            return correto
    
    return autor

def verificar_presenca(texto_chamada, lista_mestral):
    presentes = [vereador for vereador in lista_mestral
                 if re.search(r'\b' + re.escape(vereador) + r'\b', texto_chamada, re.IGNORECASE)]
    return presentes

def extrair_data_sessao(texto):
    match = re.search(
        r'(\d{1,2})\s+de\s+(janeiro|fevereiro|março|abril|maio|junho|julho|agosto|setembro|outubro|novembro|dezembro)\s+de\s+(\d{4})',
        texto,
        re.IGNORECASE
    )
    if match:
        dia, mes_nome, ano = match.groups()
        meses = {
            'janeiro': 1, 'fevereiro': 2, 'março': 3, 'abril': 4, 'maio': 5, 'junho': 6,
            'julho': 7, 'agosto': 8, 'setembro': 9, 'outubro': 10, 'novembro': 11, 'dezembro': 12
        }
        mes_num = meses[mes_nome.lower()]
        return f"{int(ano)}-{mes_num:02d}-{int(dia):02d}"
    return None

def extrair_autor_robusto(contexto_pl):
    """Extração robusta do autor com normalização"""
    # Padrão 1: "de autoria do vereador X"
    match1 = re.search(r'de\s+autoria\s+d[oa]\s+vereador(?:a)?\s+([\w\s]+?)(?:\s*[,.]|\s*dispõe|\s*institui|\s*autoriza|\s*cria|\s*\n)',
                       contexto_pl, re.IGNORECASE)
    if match1:
        autor = match1.group(1).strip()
        autor = re.sub(r'\s+', ' ', autor)
        return normalizar_autor(autor)

    # Padrão 2: Procurar nomes da lista mestra no contexto
    for nome_oficial, variacoes in {
        "Amanda": ["Amanda", "professora Amanda"],
        "André do Gás": ["André do Gás", "André"],
        "Bruninho Almeida": ["Bruninho Almeida", "Bruninho"],
        "Cacau Repórter": ["Cacau Repórter", "Cacau"],
        "Caio Perfister": ["Caio Perfister", "Caio Perfiste", "Caio"],
        "Calé": ["Calé"],
        "Dudu do Resgate": ["Dudu do Resgate", "Dudo Resgate", "Dudu"],
        "Diego Barbosa": ["Diego Barbosa", "Diego"],
        "Fabinho Filé": ["Fabinho Filé", "Fabinho"],
        "Fidel Faria": ["Fidel Faria", "Fidel"],
        "Igor Faraco": ["Igor Faraco", "Igor", "Ico Faraco"],
        "João Miguel": ["João Miguel", "João"],
        "Luciano Santos": ["Luciano Santos", "Luciano"],
        "Márcia Valentim": ["Márcia Valentim", "Márcia", "Marcia Valentin"],
        "Marcos Rangel": ["Marcos Rangel", "Rangel"],
        "Maurício Lopes": ["Maurício Lopes", "Maurício"],
        "Paulinho Nogueira": ["Paulinho Nogueira", "Paulinho"],
        "Sandrinho": ["Sandrinho"],
        "Totó": ["Totó", "Totó Online"],
        "Vitinho Nogueira": ["Vitinho Nogueira", "Vitinho", "Vitim Nogueira", "Vitin Nogueira"],
        "Érica Marra": ["Érica Marra", "Érica"]
    }.items():
        for variacao in variacoes:
            if re.search(r'\b' + re.escape(variacao) + r'\b', contexto_pl, re.IGNORECASE):
                return nome_oficial

    return "Autor não identificado"

# --- SCRIPT PRINCIPAL ---

try:
    arquivos_processados = 0
    pls_total = 0
    pls_com_ementa = 0

    arquivos = [f for f in os.listdir(pasta_transcricoes) if f.endswith('.txt')]
    print(f"📂 Processando {len(arquivos)} arquivos...\n")

    for nome_arquivo in arquivos:
        caminho_completo = os.path.join(pasta_transcricoes, nome_arquivo)
        with open(caminho_completo, 'r', encoding='utf-8') as f:
            conteudo = f.read()

        data_sessao = extrair_data_sessao(conteudo)
        url_youtube = extrair_url_youtube(conteudo)

        bloco_chamada_match = re.search(
            r"chamada dos vereadores\.(.*?)(Questão de ordem|Peço para que todos)",
            conteudo,
            re.DOTALL | re.IGNORECASE
        )
        vereadores_presentes = verificar_presenca(
            bloco_chamada_match.group(1) if bloco_chamada_match else "",
            LISTA_MESTRA_VEREADORES
        )

        # Captura PLs
        mencoes_pl = re.finditer(
            r'Projeto\s+de\s+[Ll]ei\s+(?:n[úº]mero|n[úº]|nº)?\s*(\d{1,3}\/\d{4})',
            conteudo,
            re.IGNORECASE
        )

        pls_processados_arquivo = set()

        for mencao in mencoes_pl:
            pl = mencao.group(1)

            if pl in pls_processados_arquivo:
                continue

            pls_processados_arquivo.add(pl)
            pls_total += 1

            posicao_mencao = mencao.start()

            # Contexto para extração
            inicio_contexto = max(0, posicao_mencao - 100)
            fim_contexto = min(len(conteudo), posicao_mencao + 2000)
            contexto_autor = conteudo[inicio_contexto:fim_contexto]

            # Extrai autor (já normalizado)
            autor = extrair_autor_robusto(contexto_autor)

            # Extrai ementa com função melhorada
            ementa = extrair_ementa_melhorada(conteudo, pl, posicao_mencao)
            if ementa != "Ementa não identificada":
                pls_com_ementa += 1

            # Determina status
            contexto_status = conteudo[posicao_mencao : posicao_mencao + 2000].lower()

            status_votacao = 'Não identificado'
            if re.search(r"em votação.*?aprovado", contexto_status):
                status_votacao = 'Aprovado (Votação Simbólica)'
            elif re.search(r"encaminhado.*?comissões|parecer favorável", contexto_status):
                status_votacao = 'Encaminhado para Comissão'
            elif "em discussão" in contexto_status:
                status_votacao = 'Em Discussão'
            elif 'rejeitado' in contexto_status:
                status_votacao = 'Rejeitado'

            dados_extraidos.append({
                'Data Sessão': data_sessao,
                'PL': pl,
                'Autor': autor,
                'Ementa': ementa,
                'Status': status_votacao,
                'Votos': "N/A",
                'Presentes': ", ".join(vereadores_presentes) if vereadores_presentes else "Chamada não identificada",
                'Fonte': nome_arquivo,
                'Link YouTube': url_youtube if url_youtube else "N/A"
            })

        arquivos_processados += 1
        if arquivos_processados % 10 == 0:
            print(f"✅ Processados {arquivos_processados}/{len(arquivos)} arquivos...")

    if dados_extraidos:
        df_final = pd.DataFrame(dados_extraidos).drop_duplicates(subset=['Fonte', 'PL']).reset_index(drop=True)

        df_final['Data Sessão'] = pd.to_datetime(df_final['Data Sessão'], errors='coerce')
        df_final = df_final[df_final['Data Sessão'] >= '2024-01-01']
        df_final = df_final.sort_values(by=['PL', 'Data Sessão'], ascending=[True, True])

        # Estatísticas finais
        total_pls = len(df_final)
        pls_com_ementa_final = len(df_final[df_final['Ementa'] != 'Ementa não identificada'])
        pls_com_link = len(df_final[df_final['Link YouTube'] != 'N/A'])
        percentual_ementas = (pls_com_ementa_final / total_pls) * 100 if total_pls > 0 else 0
        percentual_links = (pls_com_link / total_pls) * 100 if total_pls > 0 else 0
        
        # Calcula tamanho médio das ementas
        ementas_validas = df_final[df_final['Ementa'] != 'Ementa não identificada']['Ementa']
        tamanho_medio = ementas_validas.str.len().mean() if not ementas_validas.empty else 0

        print("\n" + "="*80)
        print("✅ ANÁLISE COMPLETA CONCLUÍDA - VERSÃO MELHORADA!")
        print("="*80)
        print(f"📊 Estatísticas:")
        print(f"   📂 Arquivos processados: {arquivos_processados}")
        print(f"   📋 Total de registros: {total_pls}")
        print(f"   📝 PLs com ementa: {pls_com_ementa_final} ({percentual_ementas:.1f}%)")
        print(f"   📏 Tamanho médio das ementas: {tamanho_medio:.0f} caracteres")
        print(f"   🎥 PLs com link YouTube: {pls_com_link} ({percentual_links:.1f}%)")
        print(f"   📅 Período: {df_final['Data Sessão'].min():%d/%m/%Y} até {df_final['Data Sessão'].max():%d/%m/%Y}")
        print(f"   🏛️ PLs únicos: {df_final['PL'].nunique()}")
        print(f"   👥 Autores únicos: {df_final['Autor'].nunique()}")

        print("\n📋 Amostra de PLs com ementas COMPLETAS:")
        for idx, row in df_final[df_final['Ementa'].str.len() > 30].head(10).iterrows():
            print(f"\n   PL {row['PL']} - {row['Autor']}")
            print(f"   📝 {row['Ementa'][:150]}...")
            if row['Link YouTube'] != 'N/A':
                print(f"   🎥 {row['Link YouTube']}")

        display(df_final)

        # Salva os arquivos
        caminho_csv = '/content/drive/MyDrive/observatorio_teresopolis/csv/base_observatorio_teresopolis_COM_EMENTAS.csv'
        caminho_excel = '/content/drive/MyDrive/observatorio_teresopolis/csv/base_observatorio_teresopolis_COM_EMENTAS.xlsx'

        df_final.to_csv(caminho_csv, index=False)

        with pd.ExcelWriter(caminho_excel, engine='openpyxl') as writer:
            df_final.to_excel(writer, sheet_name='PLs', index=False)

        print(f"\n💾 Arquivos salvos:")
        print(f"   📄 {caminho_csv}")
        print(f"   📊 {caminho_excel}")

    else:
        print("\n❌ Nenhum dado foi extraído")

except FileNotFoundError:
    print(f"❌ ERRO: A pasta '{pasta_transcricoes}' não foi encontrada.")
except Exception as e:
    print(f"❌ Erro inesperado: {e}")
    import traceback
    traceback.print_exc()

print("\n✅ PROCESSAMENTO CONCLUÍDO!")